# MedSeg 全流程测试 — VOC2012 数据集

本 Notebook 以 Pascal VOC 2012 为测试数据集，跑通 MedSeg 完整链路：

1. **数据集导入** — 每个 VOC 类别导入 10 张图像（含专家标注掩码）到 RAG 索引
2. **RAG 检索** — 输入查询图片，DINOv3 + FAISS 返回 Top-k 相似样例
3. **SAM3 memory bank 分割** — 将检索样例写入 memory bank 后锁定，对目标帧推理

## 前置准备
- 下载 [Pascal VOC 2012](http://host.robots.ox.ac.uk/pascal/VOC/voc2012/) 数据集并解压
- 准备 SAM3 模型权重文件
- GPU/CUDA 环境可用

## 0. 配置参数

修改以下路径和设备 ID 为你本机的实际值后运行。

In [ ]:
# ========================== 路径配置 ==========================
VOC2012_ROOT   = "/path/to/VOC2012"               # VOC2012 根目录（含 JPEGImages/SegmentationClass）
SAM3_CKPT      = "/path/to/sam3_checkpoint.pt"    # SAM3 模型权重
SAM3_VERSION   = "sam3.1"                          # "sam3" 或 "sam3.1"
RAG_INDEX_DIR  = "./voc2012_rag_index"             # RAG 索引持久化目录

# ========================== GPU 配置 ==========================
CUDA_RAG = 0   # RAG 系统 (DINOv3 embedder) 使用的 CUDA 设备
CUDA_SAM = 0   # SAM3 分割器使用的 CUDA 设备

# ========================== 分割参数 ==========================
TOP_K              = 5      # RAG 检索 top-k
IMAGES_PER_CLASS   = 10     # 每类别导入图像数
OUTPUT_PROB_THRESH = 0.5    # 二值化阈值
LOCK_MEMORY        = True   # 是否锁定 memory bank
QUERY_IMAGE_PATH   = None   # 设为 None 则从导入数据中自动选取一张作为查询图

import os, sys, random
from pathlib import Path

assert Path(VOC2012_ROOT).is_dir(), f"VOC2012 目录不存在: {VOC2012_ROOT}"

# VOC 全部 20 个前景类别
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
]

print(f"VOC2012 root : {VOC2012_ROOT}")
print(f"CUDA device  : RAG={CUDA_RAG}, SAM={CUDA_SAM}")
print(f"SAM3 version : {SAM3_VERSION}")
print(f"RAG index    : {RAG_INDEX_DIR}")
print(f"Per class    : {IMAGES_PER_CLASS} images, {len(VOC_CLASSES)} classes")

## 1. 数据集导入 — 每类别 10 张写入 RAG 索引

使用 `VOCImporter` 按类别发现 image-mask 配对，每类随机采样 10 张，通过 `RAGSystem.index_images()` 批量写入索引。

**类别级 mask**：`VOCImporter` 的 `class_name` 参数会自动将 VOC 调色板 mask 转换为该类的二值 PNG（0/255），与 SAM3 的 mask 输入格式对齐。

In [ ]:
from rag import RAGSystem
from rag.dataset_importers import VOCImporter

os.makedirs(RAG_INDEX_DIR, exist_ok=True)

rag = RAGSystem(
    index_dir=RAG_INDEX_DIR,
    embedder_device=f"cuda:{CUDA_RAG}",
)

total_indexed = 0
query_pool = []          # [(image_path, mask_path, class_name), ...]

for cls_idx, cls_name in enumerate(VOC_CLASSES):
    importer = VOCImporter()
    all_pairs = importer.discover_pairs(VOC2012_ROOT, class_name=cls_name)

    if len(all_pairs) == 0:
        print(f"  [{cls_idx+1:2d}/{len(VOC_CLASSES)}] {cls_name:15s} — 无可用样本，跳过")
        continue

    n_sample = min(IMAGES_PER_CLASS, len(all_pairs))
    selected = random.sample(all_pairs, n_sample)
    images, masks = zip(*selected)

    rag.index_images(list(images), list(masks))

    # 每类保留一张作为潜在查询图
    query_pool.append((images[0], masks[0], cls_name))

    total_indexed += n_sample
    print(f"  [{cls_idx+1:2d}/{len(VOC_CLASSES)}] {cls_name:15s} → {n_sample:3d}/{len(all_pairs):4d} 张")

print(f"\
索引完成: {total_indexed} 张图像, {rag.get_indexed_count()} 条向量")

## 2. 选取查询图片

若未手动指定 `QUERY_IMAGE_PATH`，则从已导入数据中随机选取一张（VOC 原图），并将其从 RAG 索引中移除，保证查询图不在检索结果中。

In [ ]:
if QUERY_IMAGE_PATH is None:
    img, msk, lbl = random.choice(query_pool)
    query_img = img
    query_cls = lbl
    # 将原图从索引中移除；但保留按类提取的 mask 对应条目
    rag.remove_image(query_img)
else:
    query_img = QUERY_IMAGE_PATH
    query_cls = "unknown"

print(f"Query image : {query_img}")
print(f"Query class : {query_cls}")
print(f"Query exists: {Path(query_img).exists()}")

## 3. RAG 检索 Top-k

DINOv3 提取查询图 CLS token（L2 归一化），FAISS IndexFlatL2 做精确 L2 检索。
返回结果包含原图路径、该类别的二值 mask 路径和 L2 距离。

In [ ]:
topk_results = rag.search(query_image=query_img, k=TOP_K)

print(f"检索到 {len(topk_results)} 条 Top-k 结果:\
")
for i, (img_p, msk_p, dist) in enumerate(topk_results):
    cls = Path(msk_p).stem.rsplit('_class', 1)
    print(f"  [{i}] dist={dist:.4f}")
    print(f"      image: {img_p}")
    print(f"      mask : {msk_p}")
    print(f"      mask exists: {Path(msk_p).exists()}")

## 4. SAM3 memory bank 分割

核心流程：
1. 将 Top-k 样例 + 查询图物化为合成视频帧序列 `0.jpg...N.jpg`
2. Top-k 帧逐一 `add_new_mask`，`propagate_in_video_preflight(run_mem_encoder=True)` 编码进 memory bank
3. 目标帧以 `run_mem_encoder=False` 推理 — **memory bank 锁定，只读**

输出：`*_mask.png`（单通道 8-bit，0/255）+ 对应的 `*.json` 元信息文件。

In [ ]:
from seg_pipeline import segment_image
from seg_pipeline.sam3_memory_segmenter import Sam3BuildConfig

sam3_cfg = Sam3BuildConfig(
    version=SAM3_VERSION,
    checkpoint_path=SAM3_CKPT,
    cuda_device=CUDA_SAM,
    compile=False,
    warm_up=False,
)

result = segment_image(
    query_image_path=query_img,
    retrieval_topk=topk_results,
    sam3_build_config=sam3_cfg,
    output_prob_thresh=OUTPUT_PROB_THRESH,
    lock_memory=LOCK_MEMORY,
)

print(f"Output mask : {result.output_mask_path}")
print(f"Meta file   : {result.meta_path}")

## 5. 查看结果

左侧为查询原图，右侧为 SAM3 输出的分割掩码。

In [ ]:
import json
from PIL import Image
from IPython.display import display, HTML

# --- 原图 ---
query_display = Image.open(query_img).convert("RGB")
query_display.thumbnail((400, 400))

# --- 分割掩码 ---
mask_img = Image.open(result.output_mask_path)
mask_img.thumbnail((400, 400))

print("查询原图 (左)  vs  SAM3 输出掩码 (右):")
display(query_display, mask_img)

# --- 元信息摘要 ---
with open(result.meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

print("\
===== 元信息摘要 =====")
print(f"usable Top-k  : {meta['retrieval']['usable_count']}")
print(f"skipped        : {len(meta['retrieval']['skipped'])}")
print(f"memory locked  : {meta['memory_locked']}")
print(f"SAM3 version   : {meta['sam3_meta']['sam3']['version']}")
print(f"prob threshold : {meta['sam3_meta']['output_prob_thresh']}")

## 6. （可选）索引持久化与复用

RAG 索引已持久化到 `RAG_INDEX_DIR`。下次使用时可直接加载，无需重新导入数据集：

```python
rag = RAGSystem(index_dir="./voc2012_rag_index")
topk = rag.search(query_image=query_img, k=5)
```